# 위험 고객 분류 + 캠페인 행동 모델

이 노트북이 만드는 파일
- `data/processed/risk_segments.csv` · `risk_segment_summary.csv` → 앱 **고객 매칭** 화면(pages/2_matching.py)에서 사용
- `models/behavior_model.joblib` → Starbucks 데이터로 학습한 캠페인 반응(행동 변화) 모델

필요한 파일 (먼저 실행할 노트북)
- `data/processed/kkbox_scored.csv`, `models/lgbm_final.joblib` ← 05_ml_comparison
- `data/processed/cluster_features_all.csv`, `models/cluster_model.joblib` ← customer_clustering_x_all
- `data/raw/starbucks/*.json` (행동 모델 부분만)


## 1. 위험 고객 분류 (K-Means 4개 군집)

In [1]:
import numpy as np
import pandas as pd
import joblib

# KKBox 점수·이탈 모델
kk = pd.read_csv("../../data/processed/kkbox_scored.csv")
saved = joblib.load("../../models/lgbm_final.joblib")

churn_model = saved["model"]
TH = saved["threshold"]
CHURN_FEATS = saved["features"]
CHURN_CATS = saved["categories"]

# 위험 고객 표시: 예전 kkbox_scored.csv에는 risk 컬럼이 없을 수 있어 여기서 계산
if "risk" not in kk.columns:
    kk["risk"] = (kk["churn_prob"] >= TH).astype(int)   # 최종 모델 기준선(약 0.2824) 이상

for c in CHURN_CATS:
    kk[c] = kk[c].astype("category")

# 팀원 군집 모델·군집용 피처
cb = joblib.load("../../models/cluster_model.joblib")
cf = pd.read_csv("../../data/processed/cluster_features_all.csv")


kk.shape, cf.shape, TH, len(cb["features"])

((100000, 37), (100000, 19), 0.2823849351481365, 18)

In [2]:
risk_users = kk[kk["risk"] == 1].copy()

Xc = risk_users[["msno"]].merge(cf, on="msno", how="left")
Xc.index = risk_users.index

feat = cb["features"]
ok = Xc[feat].notna().all(axis=1)

Xt = Xc.loc[ok, feat].copy()
for c in cb["log_features"]:
    Xt[c] = np.log1p(Xt[c])

Xs = pd.DataFrame(cb["scaler"].transform(Xt), columns=feat, index=Xt.index)
Xs = Xs.drop(columns=cb["drop_after_scale"])

risk_users["segment"] = "미분류"
risk_users.loc[Xs.index, "segment"] = pd.Series(cb["kmeans"].predict(Xs), index=Xs.index).map(cb["names"])

risk_users["segment"].value_counts()

segment
저활동·단기 구독형        3358
장기 플랜·고결제 고위험형    1748
미분류                812
장기 관계·고빈도 거래형      792
반복 거래·취소 위험형       679
Name: count, dtype: int64

In [3]:
SEGMENT_NAMES = [
    "저활동·단기 구독형",
    "반복 거래·취소 위험형",
    "장기 플랜·고결제 고위험형",
    "장기 관계·고빈도 거래형",
]

summary = risk_users[risk_users["segment"].isin(SEGMENT_NAMES)].groupby("segment").agg(
    인원=("msno", "size"),
    평균_이탈확률=("churn_prob", "mean"),
    실제_이탈률=("is_churn", "mean"),
    평균_활동일=("activity_days", "mean"),
    마지막접속후_평균일수=("days_since_last_log", "mean"),
    평균_해지횟수=("cancel_count", "mean"),
    평균_결제금액=("avg_payment", "mean"),
).reindex(SEGMENT_NAMES).round(3)
summary

,인원,평균_이탈확률,실제_이탈률,평균_활동일,마지막접속후_평균일수,평균_해지횟수,평균_결제금액
segment,,,,,,,
저활동·단기 구독형,3358,0.436,0.460,25.818,2.992,0.013,200.374
반복 거래·취소 위험형,679,0.722,0.748,42.212,12.756,1.383,125.154
장기 플랜·고결제 고위험형,1748,0.823,0.863,60.412,1.886,0.004,1020.905
장기 관계·고빈도 거래형,792,0.564,0.587,51.658,1.706,0.937,185.239


### 앱용 파일 저장

In [4]:
risk_segments = risk_users[["msno", "segment", "churn_prob"]].reset_index(drop=True)

risk_segment_summary = summary.drop(columns=["실제_이탈률"]).reset_index()

risk_segments.to_csv("../../data/processed/risk_segments.csv", index=False, encoding="utf-8-sig")
risk_segment_summary.to_csv("../../data/processed/risk_segment_summary.csv", index=False, encoding="utf-8-sig")

risk_segments.shape, risk_segments["segment"].value_counts()



((7389, 3),
 segment
 저활동·단기 구독형        3358
 장기 플랜·고결제 고위험형    1748
 미분류                812
 장기 관계·고빈도 거래형      792
 반복 거래·취소 위험형       679
 Name: count, dtype: int64)

## 2. Starbucks 데이터로 캠페인 반응(행동) 모델 학습

In [5]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_poisson_deviance,
    mean_squared_error,
)
from sklearn.model_selection import GroupKFold, cross_val_predict


STARBUCKS_DIR = Path("../../data/raw/starbucks")
MODEL_DIR = Path("../../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

PORTFOLIO_PATH = STARBUCKS_DIR / "portfolio.json"
PROFILE_PATH = STARBUCKS_DIR / "profile.json"
TRANSCRIPT_PATH = STARBUCKS_DIR / "transcript.json"

for path in [PORTFOLIO_PATH, PROFILE_PATH, TRANSCRIPT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {path}")

print("Starbucks 데이터 경로 확인 완료")

Starbucks 데이터 경로 확인 완료


In [6]:
portfolio = pd.read_json(
    PORTFOLIO_PATH,
    orient="records",
    lines=True,
)

profile = pd.read_json(
    PROFILE_PATH,
    orient="records",
    lines=True,
)

transcript = pd.read_json(
    TRANSCRIPT_PATH,
    orient="records",
    lines=True,
)

print("portfolio :", portfolio.shape)
print("profile   :", profile.shape)
print("transcript:", transcript.shape)

display(portfolio)
display(profile.head())
display(transcript.head())

portfolio : (10, 6)
profile   : (17000, 5)
transcript: (306534, 4)


,reward,channels,difficulty,duration,offer_type,id
0,10,"[email, mobile, social]",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"[web, email, mobile, social]",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"[web, email, mobile]",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"[web, email, mobile]",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"[web, email]",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7
5,3,"[web, email, mobile, social]",7,7,discount,2298d6c36e964ae4a3e7e9706d1fb8c2
6,2,"[web, email, mobile, social]",10,10,discount,fafdcd668e3743c1bb461111dcafc2a4
7,0,"[email, mobile, social]",0,3,informational,5a8bc65990b245e5a138643cd4eb9837
8,5,"[web, email, mobile, social]",5,5,bogo,f19421c1d4aa40978ebb69ca19b0e20d
9,2,"[web, email, mobile]",10,7,discount,2906b810c7d4411798c6938adc9daaa5


,gender,age,id,became_member_on,income
0,None,118,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN
1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0
2,None,118,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN
3,F,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0
4,None,118,a03223e636434f42ac4c3df47e8bac43,20170804,NaN


,person,event,value,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0


In [7]:
CHANNELS = ["email", "mobile", "social", "web"]

for channel in CHANNELS:
    portfolio[channel] = portfolio["channels"].apply(
        lambda channels: int(channel in channels)
    )

# 고객이 달성해야 하는 구매 조건 대비 보상 비율
portfolio["benefit"] = np.where(
    portfolio["difficulty"] > 0,
    portfolio["reward"] / portfolio["difficulty"],
    0.0,
)

portfolio_check = portfolio[
    [
        "id",
        "offer_type",
        "reward",
        "difficulty",
        "benefit",
        "duration",
        *CHANNELS,
    ]
].copy()

display(portfolio_check)

,id,offer_type,reward,difficulty,benefit,duration,email,mobile,social,web
0,ae264e3637204a6fb9bb56bc8210ddfd,bogo,10,10,1.000000,7,1,1,1,0
1,4d5c57ea9a6940dd891ad53e9dbe8da0,bogo,10,10,1.000000,5,1,1,1,1
2,3f207df678b143eea3cee63160fa8bed,informational,0,0,0.000000,4,1,1,0,1
3,9b98b8c7a33c4b65b9aebfe6a799e6d9,bogo,5,5,1.000000,7,1,1,0,1
4,0b1e1539f2cc45b7b9fa7c272da2e1d7,discount,5,20,0.250000,10,1,0,0,1
5,2298d6c36e964ae4a3e7e9706d1fb8c2,discount,3,7,0.428571,7,1,1,1,1
6,fafdcd668e3743c1bb461111dcafc2a4,discount,2,10,0.200000,10,1,1,1,1
7,5a8bc65990b245e5a138643cd4eb9837,informational,0,0,0.000000,3,1,1,1,0
8,f19421c1d4aa40978ebb69ca19b0e20d,bogo,5,5,1.000000,5,1,1,1,1
9,2906b810c7d4411798c6938adc9daaa5,discount,2,10,0.200000,7,1,1,0,1


In [8]:
    # 118세는 Starbucks 데이터에서 결측치를 나타내는 값
profile["age"] = profile["age"].replace(118, np.nan)

profile["became_member_on"] = pd.to_datetime(
    profile["became_member_on"].astype(str),
    format="%Y%m%d",
)

# 가장 최근 가입일을 기준으로 가입기간 계산
reference_date = profile["became_member_on"].max()

profile["tenure_days"] = (
    reference_date - profile["became_member_on"]
).dt.days

profile[
    ["age", "gender", "tenure_days"]
].describe(include="all")

,age,gender,tenure_days
count,14825.000000,14825,17000.000000
unique,NaN,3,NaN
top,NaN,M,NaN
freq,NaN,8484,NaN
mean,54.393524,NaN,517.449882
std,17.383705,NaN,411.223904
min,18.000000,NaN,0.000000
25%,42.000000,NaN,208.000000
50%,55.000000,NaN,358.000000
75%,66.000000,NaN,791.000000


In [9]:
transcript_value = pd.json_normalize(transcript["value"])

empty_offer_id = pd.Series(
    pd.NA,
    index=transcript_value.index,
    dtype="object",
)

offer_id_space = (
    transcript_value["offer id"]
    if "offer id" in transcript_value.columns
    else empty_offer_id
)

offer_id_under = (
    transcript_value["offer_id"]
    if "offer_id" in transcript_value.columns
    else empty_offer_id
)

transcript["offer_id"] = offer_id_space.combine_first(offer_id_under)

transcript["amount"] = (
    transcript_value["amount"]
    if "amount" in transcript_value.columns
    else np.nan
)

transcript["day"] = transcript["time"] / 24

display(transcript["event"].value_counts())
print(
    "캠페인 발송 시점:",
    sorted(
        transcript.loc[
            transcript["event"] == "offer received",
            "time",
        ].unique()
    ),
)

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64

캠페인 발송 시점: [np.int64(0), np.int64(168), np.int64(336), np.int64(408), np.int64(504), np.int64(576)]


In [10]:
# 캠페인 발송 이후 행동을 관찰하는 시간
W = 72  # 72시간 = 3일

waves = sorted(
    transcript.loc[
        transcript["event"] == "offer received",
        "time",
    ].unique()
)

received = transcript.loc[
    transcript["event"] == "offer received",
    ["person", "time", "offer_id"],
].copy()

transactions = transcript.loc[
    transcript["event"] == "transaction",
    ["person", "time", "amount"],
].copy()

panel_rows = []

for wave in waves:
    # 모든 고객을 해당 발송 회차의 기본 대상으로 생성
    wave_panel = profile[["id"]].rename(
        columns={"id": "person"}
    )
    wave_panel["wave"] = wave

    # 해당 시점에 실제로 받은 캠페인
    wave_offer = (
        received.loc[
            received["time"] == wave,
            ["person", "offer_id"],
        ]
        .drop_duplicates("person")
    )

    wave_panel = wave_panel.merge(
        wave_offer,
        on="person",
        how="left",
    )

    # 발송 후 72시간 동안의 거래
    wave_transactions = (
        transactions.loc[
            (transactions["time"] >= wave)
            & (transactions["time"] < wave + W)
        ]
        .groupby("person")
        .agg(
            n_tx=("amount", "size"),
            spend=("amount", "sum"),
        )
        .reset_index()
    )

    wave_panel = wave_panel.merge(
        wave_transactions,
        on="person",
        how="left",
    )

    wave_panel[["n_tx", "spend"]] = (
        wave_panel[["n_tx", "spend"]].fillna(0)
    )

    panel_rows.append(wave_panel)

panel = pd.concat(
    panel_rows,
    ignore_index=True,
)

print("기본 패널:", panel.shape)
display(panel.head())

기본 패널: (102000, 5)


,person,wave,offer_id,n_tx,spend
0,68be06ca386d4c31939f3a4f0e3dd783,0,NaN,0.0,0.00
1,0610b486422d4921ae7d2bf64640c50b,0,NaN,1.0,21.51
2,38fe809add3b4fcf9315a9694bb96ff5,0,NaN,0.0,0.00
3,78afa995795e4d85b5d9ceeca43f5fef,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,0.0,0.00
4,a03223e636434f42ac4c3df47e8bac43,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,0.0,0.00


In [11]:
offer_columns = [
    "id",
    "offer_type",
    "email",
    "mobile",
    "social",
    "web",
    "benefit",
    "duration",
]

panel = (
    panel.merge(
        portfolio[offer_columns],
        left_on="offer_id",
        right_on="id",
        how="left",
    )
    .drop(columns="id")
)

# 캠페인을 받지 않은 고객
panel["offer_type"] = panel["offer_type"].fillna("none")

offer_value_columns = [
    "email",
    "mobile",
    "social",
    "web",
    "benefit",
    "duration",
]

panel[offer_value_columns] = (
    panel[offer_value_columns].fillna(0)
)

# income은 의도적으로 결합하지 않음
panel = (
    panel.merge(
        profile[
            [
                "id",
                "age",
                "gender",
                "tenure_days",
            ]
        ],
        left_on="person",
        right_on="id",
        how="left",
    )
    .drop(columns="id")
)

print("최종 패널:", panel.shape)

panel_summary = (
    panel.groupby(
        "offer_type",
        observed=True,
    )
    .agg(
        데이터수=("person", "size"),
        평균_거래횟수=("n_tx", "mean"),
        평균_거래금액=("spend", "mean"),
    )
    .round(3)
)

display(panel_summary)

최종 패널: (102000, 15)


,데이터수,평균_거래횟수,평균_거래금액
offer_type,,,
bogo,30499,1.046,13.397
discount,30543,0.962,12.612
informational,15235,0.900,11.281
none,25723,0.594,7.575


In [12]:
BEHAVIOR_CATS = [
    "gender",
    "offer_type",
]

BEHAVIOR_CUSTOMER_FEATS = [
    "age",
    "gender",
    "tenure_days",
]

BEHAVIOR_OFFER_FEATS = [
    "offer_type",
    "email",
    "mobile",
    "social",
    "web",
    "benefit",
    "duration",
]

BEHAVIOR_FEATS = (
    BEHAVIOR_CUSTOMER_FEATS
    + BEHAVIOR_OFFER_FEATS
)

for column in BEHAVIOR_CATS:
    panel[column] = panel[column].astype("category")

print("행동 모델 피처:", BEHAVIOR_FEATS)
print("범주형 피처:", BEHAVIOR_CATS)

display(
    panel[
        BEHAVIOR_FEATS + ["n_tx"]
    ].head()
)

행동 모델 피처: ['age', 'gender', 'tenure_days', 'offer_type', 'email', 'mobile', 'social', 'web', 'benefit', 'duration']
범주형 피처: ['gender', 'offer_type']


,age,gender,tenure_days,offer_type,email,mobile,social,web,benefit,duration,n_tx
0,NaN,NaN,529,none,0.0,0.0,0.0,0.0,0.00,0.0,0.0
1,55.0,F,376,none,0.0,0.0,0.0,0.0,0.00,0.0,1.0
2,NaN,NaN,14,none,0.0,0.0,0.0,0.0,0.00,0.0,0.0
3,75.0,F,443,bogo,1.0,1.0,0.0,1.0,1.00,7.0,0.0
4,NaN,NaN,356,discount,1.0,0.0,0.0,1.0,0.25,10.0,0.0


In [13]:
behavior_model = LGBMRegressor(
    objective="poisson",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=50,
    random_state=42,
    verbosity=-1,
)

group_kfold = GroupKFold(
    n_splits=5
)

oof_prediction = cross_val_predict(
    behavior_model,
    panel[BEHAVIOR_FEATS],
    panel["n_tx"],
    cv=group_kfold,
    groups=panel["person"],
)

# Poisson deviance 계산을 위해 0보다 크게 제한
oof_prediction = np.clip(
    oof_prediction,
    1e-8,
    None,
)

mae = mean_absolute_error(
    panel["n_tx"],
    oof_prediction,
)

rmse = np.sqrt(
    mean_squared_error(
        panel["n_tx"],
        oof_prediction,
    )
)

poisson_deviance = mean_poisson_deviance(
    panel["n_tx"],
    oof_prediction,
)

print(f"GroupKFold MAE             : {mae:.4f}")
print(f"GroupKFold RMSE            : {rmse:.4f}")
print(f"GroupKFold Poisson deviance: {poisson_deviance:.4f}")

GroupKFold MAE             : 0.7747
GroupKFold RMSE            : 0.9948
GroupKFold Poisson deviance: 1.1694


In [14]:
behavior_validation = (
    panel.assign(
        predicted_n_tx=oof_prediction
    )
    .groupby(
        "offer_type",
        observed=True,
    )
    .agg(
        데이터수=("person", "size"),
        실제_평균거래=("n_tx", "mean"),
        예측_평균거래=("predicted_n_tx", "mean"),
    )
)

behavior_validation["차이"] = (
    behavior_validation["예측_평균거래"]
    - behavior_validation["실제_평균거래"]
)

display(
    behavior_validation.round(3)
)

,데이터수,실제_평균거래,예측_평균거래,차이
offer_type,,,,
bogo,30499,1.046,1.045,-0.001
discount,30543,0.962,0.958,-0.003
informational,15235,0.900,0.902,0.002
none,25723,0.594,0.598,0.004


In [15]:
behavior_model.fit(
    panel[BEHAVIOR_FEATS],
    panel["n_tx"],
)

print("행동 모델 최종 학습 완료")

행동 모델 최종 학습 완료


In [16]:
behavior_categories = {
    column: panel[column].cat.categories.tolist()
    for column in BEHAVIOR_CATS
}

behavior_bundle = {
    "model": behavior_model,
    "features": BEHAVIOR_FEATS,
    "categorical_features": BEHAVIOR_CATS,
    "categories": behavior_categories,
    "target": "n_tx",
    "observation_hours": W,
}

BEHAVIOR_MODEL_PATH = (
    MODEL_DIR / "behavior_model.joblib"
)

joblib.dump(
    behavior_bundle,
    BEHAVIOR_MODEL_PATH,
)

print(
    f"행동 모델 저장 완료: {BEHAVIOR_MODEL_PATH}"
)
print("저장 피처:", behavior_bundle["features"])
print("저장 범주:", behavior_bundle["categories"])

행동 모델 저장 완료: ..\..\models\behavior_model.joblib
저장 피처: ['age', 'gender', 'tenure_days', 'offer_type', 'email', 'mobile', 'social', 'web', 'benefit', 'duration']
저장 범주: {'gender': ['F', 'M', 'O'], 'offer_type': ['bogo', 'discount', 'informational', 'none']}


In [17]:
loaded_behavior = joblib.load(
    BEHAVIOR_MODEL_PATH
)

assert (
    loaded_behavior["features"]
    == BEHAVIOR_FEATS
)

assert set(
    loaded_behavior["categories"]
) == set(BEHAVIOR_CATS)

test_prediction = loaded_behavior[
    "model"
].predict(
    panel[BEHAVIOR_FEATS].head(10)
)

print("모델 재로딩 확인 완료")
print("예측값:", test_prediction.round(3))

모델 재로딩 확인 완료
예측값: [0.597 0.598 0.444 0.873 0.534 0.422 0.741 0.7   0.486 0.595]
